# Two networks

The model follows the aggregation-function approach. The networks are trained separately and composed only at prediction time.

1. **Criteria network.** User and item embeddings of size 64 are concatenated and passed through ReLU layers `[128, 64, 32, 16, 8]`, then 7 ReLU outputs.
2. **Overall network.** The seven criteria are z-scored with the fit-split mean and standard deviation and passed through ReLU layers `[64, 32, 16, 8]` to one overall rating.

The overall network is trained on the true criteria, not on the other network's predictions. At test time those predictions are z-scored with the same fit statistics and fed to the overall network.

Weights are drawn from Normal(0, 0.05). Output biases are then shifted by +1. A negative ReLU bias has no gradient, and without the shift three of the seven criteria outputs stay at 0.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from src.data import load_ratings
from src.models import OverallDNN, criteria_dnn, single_dnn
from src.experiment import overfit_criteria, sample_overfit_rows


## Architectures

In [2]:
ratings = load_ratings(ROOT / 'texts.rar')
criteria = criteria_dnn(ratings.n_users, ratings.n_items)
overall = OverallDNN()
single = single_dnn(ratings.n_users, ratings.n_items)
for name, model in [('criteria', criteria), ('overall', overall), ('single', single)]:
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{name:<10} {n_params:>10} parameters')
print(criteria)
print(overall)


criteria      4761591 parameters
overall          3265 parameters
single       19373953 parameters
EmbeddingDNN(
  (user_emb): Embedding(72119, 64)
  (item_emb): Embedding(1850, 64)
  (mlp): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): ReLU()
    (8): Linear(in_features=16, out_features=8, bias=True)
    (9): ReLU()
    (10): Linear(in_features=8, out_features=7, bias=True)
    (11): ReLU()
  )
)
OverallDNN(
  (mlp): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=8, bias=True)
    (7): ReLU(

## Overfit check

Before the 5-fold run, the criteria network has to memorize a few dozen rows. This is the same check `python -m src.experiment` runs first. The full comparison is the logged run in `results/experiment.log`.

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
user, item, criteria_y = sample_overfit_rows(ratings, n_rows=48, seed=42)
result = overfit_criteria(
    user, item, criteria_y,
    ratings.n_users, ratings.n_items,
    device=device,
    max_epochs=30,
    patience=5,
    batch_size=512,
    base_seed=42,
)
print(device)
print(result)


  epoch 01  val_mae 3.0927


  epoch 02  val_mae 3.0891


  epoch 03  val_mae 3.0854


  epoch 04  val_mae 3.0818


  epoch 05  val_mae 3.0780


...

  epoch 28  val_mae 0.6802


  epoch 29  val_mae 0.5609


  epoch 30  val_mae 0.5959


overfit seed 42: train MAE 3.0964 -> 0.5609 (drop 2.5355)


cuda
{'seed': 42, 'initial_mae': 3.0963821411132812, 'final_mae': 0.5609182715415955, 'drop': 2.535463869571686, 'rows': 48, 'batch_size': 16, 'max_epochs': 30}
